# Step 21.3 — Temperature：分清概率缩放与数值检查

在仓库根目录先同步母版：

```bash
git pull
```

再建立不覆盖已有笔记的独立工作副本（仓库已有 notebooks/work 目录）：

```bash
cp -n notebooks/94_step21_3_temperature.ipynb notebooks/work/94_step21_3_temperature_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上命令仅供未来使用，本次未执行。沿用项目 Python 3.12 / .venv，API 固定 **MLX v0.32.2**。

2026-11-17 教学修订：**21.3 累计 2/2＝历史分组柱图修订 1 次＋本次讲解修订 1 次**，不重置历史。现有三段代码、分组柱图说明与练习保留；只补核心缩放、检查边界和接口衔接。

所有代码单元 execution_count=null、outputs=[]。本次没有运行 MLX、Notebook、采样、绘图、测试、训练、实验或语料／checkpoint 操作；纸算和未来观察计划不是已取得的输出。


## 1. 温度改变选择分布，不更新模型

[21.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/2dd833d835c3556784699ca9ebb9f5e3a7713666/notebooks/93_step21_2_greedy_vs_sampling.ipynb)从同一行 logits 比较 greedy 与 sampling。本节在选择之前加入 **temperature（温度）τ**：调整分数差的尺度，保持原 token ID，不修改模型权重或 checkpoint。

温度小于 1 时，高分候选相对更占优势；大于 1 时，候选间的概率差趋于缩小。这是对给定分数的可解释变换，不是“创造力”或事实正确性的开关。

流程仍是末位置 logits → 温度变换 → greedy 或 sampling。低温下的 sampling 仍是按分布选择；greedy 则单独取 argmax。本节不使用 τ=0 来伪装 greedy，因为除以 0 没有这里需要的定义。


## 2. 用相对 odds 看为什么概率会变

对有限 logits z 和正温度 τ：

$$
p_i(\tau)=\frac{\exp(z_i/\tau)}{\sum_j\exp(z_j/\tau)},\qquad
\frac{p_i(\tau)}{p_j(\tau)}
=\exp\!\left(\frac{z_i-z_j}{\tau}\right).
$$

这里的 relative odds 指两个候选的相对概率 pᵢ/pⱼ。右式来自约去相同的归一化分母：真正决定相对优势的是分数差除以 τ。若 zᵢ>zⱼ，降温会放大这个正差，升温会缩小它。

设每行最大值为 m，使用 sᵢ=(zᵢ−m)/τ 与直接 zᵢ/τ 得到相同的理论 softmax，因为所有指数权重都乘了同一个 exp(−m/τ)，归一化时抵消。先减最大值再除温度，也避免先把大的正 logit 放大后再去处理它。

在精确算术下：

$$
s_i-s_j=\frac{z_i-z_j}{\tau}.
$$

τ>0 不改变差的正负，所以**唯一 argmax 的位置不变**，但候选间的相对概率会变。greedy 的唯一选择因而不变，sampling 的理论选中概率可以明显改变；并不保证少数几次随机结果按同样趋势变化。


## 3. 手算：同一个最大 ID，三种分布

沿用现有图的人工分数 z=[0,ln 2]，ID 1 分数更高。减去 m=ln 2 后，缩放分数为 [−ln 2/τ,0]。两候选的相对概率为 p₁/p₀=2^(1/τ)：

| τ | p₁/p₀ | 归一化后的 [p₀,p₁] | 唯一最大 ID |
|---:|---|---|---:|
| 0.5 | 4 | [1/5,4/5] | 1 |
| 1 | 2 | [1/3,2/3] | 1 |
| 2 | √2 | [1/(1+√2),√2/(1+√2)] | 1 |

例如 τ=0.5 时，只知道比例 1:4 还不是概率；除以总和 5，才得到 1/5 和 4/5。这些是指定分数的纸面推导，不是本次 MLX 输出。

正确顺序是变换 logits 后再归一化或抽样。不能把已得的概率直接除以 τ；那通常连总和为 1 都不满足。把概率再当作 categorical 的 logits，也会得到另一个分布。


## 4. 最短核心变换与 shape

调用者先提供具有非空词表轴 V 的 MLX logits 数组，并采用正、有限且适于当前数值范围的 τ。下面只是原函数中两行核心计算的节选，不是新增执行单元，也没有包含错误检查：

```python
z = logits.astype(mx.float32)
scaled = (z - mx.max(z, axis=-1, keepdims=True)) / tau
```

| 对象／操作 | (B,V) 输入 | 单行 (V,) 输入 |
|---|---|---|
| 转为 float32 的 z | (B,V) | (V,) |
| 沿 axis=-1 求 max，keepdims=True | (B,1) | (1,) |
| z 减该最大值 | 广播回 (B,V) | 广播回 (V,) |
| 除以标量 τ | (B,V) | (V,) |
| softmax(scaled,axis=-1) | 概率仍为 (B,V) | 概率仍为 (V,) |

keepdims=True 保留长度为 1 的候选轴，便于沿 V 广播；不是把整个 batch 混成一个最大值。函数返回的是 **scaled logits**，不是概率或 token ID；它不接收、拆分或返回随机 key。

为什么还有较长的完整接口？上面两行表达数学变换，下面的检查用来拒绝当前教学接口不支持的输入和非有限结果。不要把两者当作两个不同算法。


### 4.1 按实际执行顺序读检查

| 完整函数中的阶段 | 检查或操作 | 不应额外推断什么？ |
|---|---|---|
| 温度 Python 类型 | 只接受 int/float，显式排除 bool | 不是任意标量对象都支持 |
| 转为 Python float 后 | math.isfinite(tau) 且 tau>0 | Python 有限不保证之后 float32 运算无损 |
| logits 转 float32 并 mx.eval(z) | all(isfinite(z))，拒绝 NaN、+Inf、−Inf | 转换后的表示才是接下来使用的输入 |
| 核心缩放并 mx.eval(scaled) | 再检查输出全为有限值 | 只检查有限性，不检查概率质量或排序精度 |
| return scaled | 返回同 shape 的 float32 缩放分数 | 不自动做 softmax、sampling 或 greedy |

mx.isfinite 返回逐元素布尔数组；mx.all 汇成一个判断，.item() 把它交给 Python if。mx.eval 是这里为主机检查安排的求值边界，不是温度数学公式的一部分。

当前函数没有专门的 rank／空词表轴检查；正确的词表轴由调用者保证，非法轴等问题可能由 MLX 报错。这里不新增校验逻辑，也不把自定义 ValueError 说成 MLX 自动提供的温度策略。


In [ ]:
import math
import mlx.core as mx

# 本单元及后续所有单元均为未执行的教学源码。
# 仅在未来独立工作副本中按顺序运行；本次材料制作没有导入 MLX。

def temperature_logits(logits, temperature):
    if isinstance(temperature, bool) or not isinstance(temperature, (int, float)):
        raise ValueError("temperature must be a positive finite number")
    tau = float(temperature)
    if not math.isfinite(tau) or tau <= 0:
        raise ValueError("temperature must be positive and finite; greedy is separate")
    z = logits.astype(mx.float32)
    mx.eval(z)
    if not bool(mx.all(mx.isfinite(z)).item()):
        raise ValueError("this lesson accepts finite logits only")
    scaled = (z - mx.max(z, axis=-1, keepdims=True)) / tau
    mx.eval(scaled)
    if not bool(mx.all(mx.isfinite(scaled)).item()):
        raise ValueError("temperature scaling overflowed; choose a representable scale")
    return scaled

### 4.2 数学极限与当前接口接受范围

若唯一最大 logit 存在，数学上 τ→0⁺ 时概率集中到它；存在并列最大时，不能说极限必然只留下一个 ID。固定有限 logits 下，τ→∞ 时分布趋向均匀。**0 和 +∞ 都不是这个函数接受的温度值。**

实际 float32 还有另一层限制：

- 减最大值能帮助稳定计算，但极端的有限输入相减、或很小温度的除法，仍可能得到非有限 scaled；函数的最后检查会拒绝这种结果。它不钳制数值，也不自动改成 greedy。
- 温度或分数差在有限精度中可能舍入、下溢；若输出仍有限，检查未必发现差异已经丢失。因而“唯一 argmax 不变”是上面的精确算术结论，不能无限制扩展为所有极端 float32 输入的承诺。
- 即使 scaled 全部有限，后续 softmax 的很小概率仍可能下溢为 0；本 helper 没有在返回后检查概率。不能把数学上正概率当作有限精度下永远非零。

本函数要求转换后的**所有 logits 有限**，所以也拒绝用作 mask 的 −Inf。这里不能先 top-k 屏蔽再调用温度函数；需要与下一节组合时，按第 6 节给出的顺序衔接。


## 5. 一个未来观察：最高柱会换到另一个 token 吗？

先判断：固定 z=[0,ln 2]，把 τ 改为 0.5、1、2，最高概率的 token ID 会不会改变，而它的概率能否改变？先写出依据来自 argmax 的哪条关系，再决定是否在未来工作副本中观察。

下一个单元只计算并显示三个分布；已有分组柱图是**同一个观察的图形表达**。未来可把数值／同色两柱之和与纸算对照，并检查最高柱的 ID 及柱高变化。若出现不符，应先对照实际 logits、τ、axis 和数值表示，不能把图本身当作公式证明。

这是确定性分布计算，不是抽样频率实验。最后一行 categorical 只是原有注释，没有被调用；不需要先运行 21.2 的八次采样，也不从有限样本“验证”本节概率。本轮不执行任何单元，不生成数值或图片输出。


In [ ]:
z = mx.array([[0., math.log(2)]], dtype=mx.float32)
for tau in (0.5, 1.0, 2.0):
    scaled = temperature_logits(z, tau)
    p = mx.softmax(scaled, axis=-1)
    mx.eval(p)
    print(tau, p.tolist())
# 如未来需要抽样：mx.random.categorical(scaled, key=mx.random.key(17))

## 5.1 用 grouped bar plot 比较温度分布（仅源码，未绘制）

同一份人工 logits `z=[0, ln(2)]` 在三种温度下给出三个 categorical distributions。横轴是原始 token ID（0、1），纵轴是该 token 的理论概率，范围 0 到 1；颜色表示 temperature。每一种颜色对应的两个柱高相加应为 1，而不是要求每个 token 位置的三种颜色相加为 1。

先按公式 `softmax(z / tau)` 计算各分布，再画图。这里复用上方 `temperature_logits` 的稳定化与参数检查，`mx.eval` 之后才转换为 Python list 交给 matplotlib。它们是对给定分数的理论分布作数值计算，不是 sampling 频率、不是真实训练模型的预测质量，也不是已取得的运行测量。当前并未执行绘图，因此没有图形输出或实测结论。

Grouped bars（分组柱）让同一个 token 的三种温度并排，避免相互遮挡。`width=0.24` 是每根柱子的宽度；第 i 组使用 `(i-1)*width` 偏移，使 i=0、1、2 分别位于中心左侧、中心和右侧。`centers=[0,1]` 保留 token 的刻度位置，偏移只改变展示坐标，不改变 token ID 或概率。

阅读时先比较 token 1 在降温和升温后的柱高，再检查同色两柱之和。图可以帮助观察趋势，但不能代替前面的 odds ratio 推导；改变横轴偏移或颜色不会改变概率模型。

In [ ]:
# 未来工作副本中的绘图源码：当前未运行、无输出。
import matplotlib.pyplot as plt

plot_logits = mx.array([[0., math.log(2)]], dtype=mx.float32)
temperatures = (0.5, 1.0, 2.0)
centers = [0, 1]
width = 0.24
fig, ax = plt.subplots(figsize=(6, 4))
for i, tau in enumerate(temperatures):
    scaled = temperature_logits(plot_logits, tau)
    probabilities = mx.softmax(scaled, axis=-1)
    mx.eval(probabilities)
    heights = probabilities.tolist()[0]
    positions = [x + (i - 1) * width for x in centers]
    ax.bar(positions, heights, width=width, label=f"temperature={tau:g}")
ax.set_xticks(centers)
ax.set_xticklabels(["token 0", "token 1"])
ax.set_xlabel("Token ID")
ax.set_ylabel("Theoretical probability")
ax.set_ylim(0, 1)
ax.set_title("Temperature changes probabilities for fixed toy logits")
ax.legend()
fig.tight_layout()
plt.show()


## 6. 接口衔接：先变换分数，再选择

与 [21.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/2dd833d835c3556784699ca9ebb9f5e3a7713666/notebooks/93_step21_2_greedy_vs_sampling.ipynb)直接连接时：

| 阶段 | 输入 → 输出 | key |
|---|---|---|
| temperature_logits(last,τ) | last (B,V) → scaled (B,V) | 不使用随机 key |
| choose_greedy(scaled) | (B,V) → token (B,) int32 | 不使用随机 key |
| 或 choose_sample(scaled,key) | (B,V) → (token (B,),next_key) | 显式拆分并返回 |
| 为 append 增轴 | token[:,None] → (B,1) | sampling 还须保留返回的 next_key |

这里直接把 scaled logits 传入选择器，不先 softmax。greedy 与 sampling 是两个可选分支，不是先后各选一次。

[21.4](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/2dd833d835c3556784699ca9ebb9f5e3a7713666/notebooks/95_step21_4_top_k_sampling.ipynb)的 exact_top_k_logits 目前只接**单行 (V,)**，返回 (filtered_logits, selected_IDs)，不是 batch 接口。对于现有 B=1 的生成路径，组合形状应是：

```text
last (1,V)
 → temperature_logits → scaled (1,V)
 → 取 scaled[0] → (V,)
 → exact_top_k_logits → filtered (V,) 与 selected_IDs 列表
 → filtered[None,:] → (1,V)
 → 21.2 choose_sample → token (1,) 与 next_key
 → token[:,None] → (1,1)，再 append
```

因此先对有限 logits 做温度缩放，再交给同样要求有限输入的 top-k helper；top-k 输出的部分 −Inf 可以给 categorical，却不能回送本节的有限性检查。这里仅核对已有接口，不实现 top-k、不改 21.4，也不把单行筛选器冒充 batch 支持。

真实接入点是 logits[:, -1, :] 之后、选择器之前。不在训练交叉熵里悄悄加入生成温度，也不修改 checkpoint。温度变换保持原词表 ID；展示用的柱子偏移不改变 ID 或概率。


## 7. 练习（不附答案）

1. 手算 z=[0,ln3] 在 τ=1 与 τ=0.5 的 odds ratio。
2. 提出一个并列最大 logits 例子，解释 τ→0⁺ 与 argmax tie policy 的区别。
3. 为什么 temperature=NaN、0、负数应拒绝？为接口写出预期错误清单，不给答案代码。

请先用纸笔作答，再决定是否在工作副本中添加自己的代码。母版不填入参考答案。

## 8. 固定版本依据与材料状态

- [MLX v0.32.2 random.cpp Python bindings](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp)：categorical 接收 logits、移除候选轴与显式 key。
- [MLX v0.32.2 ops.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/ops.cpp)：max 的 axis/keepdims、isfinite、softmax 接口。

本次保留完整温度函数、数值观察代码、分组柱图说明和绘图源码，未改变代码行为或代答练习。仅静态核对 JSON/schema、合法唯一 cell ID、语法、AST 与相邻接口；所有代码单元未执行、无输出，不声称已运行、绘制、采样成功或学员掌握。

21.3 累计 **2/2**：历史分组柱图修订 1 次＋本次讲解修订 1 次，其他历史计数保持。本轮停在本节，等待独立验收。
